# ARQ

ARQ 基于 asyncio 和 Redis，适合异步 I/O 任务。任务函数是协程，天然适配 HTTP 请求、数据库访问等异步资源。

```zsh
arq python_notes.task_queues.arq.tasks.WorkerSettings
python -m python_notes.task_queues.arq.client
```


In [ ]:
from typing import Any

from arq import cron, func
from arq.connections import RedisSettings
from arq.worker import Retry


async def add(ctx: dict[str, Any], a: int, b: int) -> int:
    return a + b


async def flaky_job(ctx: dict[str, Any], value: int) -> int:
    if ctx['job_try'] < 3:
        raise Retry(defer=1)
    return value * 2


async def heartbeat(ctx: dict[str, Any]) -> None:
    print('ARQ worker heartbeat')


class WorkerSettings:
    functions = [
        func(add, name='add', keep_result=3600, timeout=10),
        func(flaky_job, name='flaky_job', keep_result=3600, timeout=15, max_tries=3),
    ]
    cron_jobs = [cron(heartbeat, minute={0, 30}, run_at_startup=True)]
    redis_settings = RedisSettings(host='localhost', port=6379, database=0)


In [ ]:
import asyncio

from arq import create_pool

from python_notes.task_queues.arq.tasks import redis_config_from_env


async def main() -> None:
    redis = await create_pool(redis_config_from_env().to_settings())
    try:
        job = await redis.enqueue_job('add', 3, 7)
        retried = await redis.enqueue_job('flaky_job', 21)
        print('Job ID:', job.job_id)
        print('Add Result:', await job.result(timeout=5))
        print('Retry Result:', await retried.result(timeout=10))
    finally:
        await redis.close()


asyncio.run(main())
